# Purpose:
- For **Fire Perimeter**:
    - Generate a circular polygon and crop the flight image based on that polygon
- For **Burn Severity**:
    - Crop the raster image based on input fire perimeter in GeoJSON (*product generated* or [CALFIRE](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-fire-perimeters-all-1/explore)) or shapefile ([MTBS](https://www.mtbs.gov/viewer/index.html))
- For **either**:
    - Reproject a raster image to another coordinate reference systems (Check for burn severity raster images [MTBS](https://www.mtbs.gov/viewer/index.html))
    
***Due to GitHub file size limit, the post-RTC data is not available on this repository. Please following the steps in the 'radiometric_terrain_correction' repository to obtain the RTC data***

Cropped RTC image is available for the application of the other 3 notebooks in this repo.


In [1]:
import sys
from pathlib import Path

# Add the path to the utils folder to sys.path
utils_path = Path('../python').resolve()
sys.path.append(str(utils_path))

from pathlib import Path
from rasterio.crs import CRS
import rasterio
from pyproj import Transformer
from shapely.geometry import box
from crop_utils import (crop_image_by_coordinates, 
                        crop_image_by_geojson_shp, 
                        reproject_geotiff)
from edit_path_utils import (edit_paths)
print(utils_path)

/home/albertpc/uavsar-cads-main/python


---
## Crop image by box from center
Returns a cropped version of the input image based on rectangle centered on coordinate

**Parameters**:
- `path_to_images` (list): a list containing the paths to the images cropping from 
- `output_names` (list): a list containing the output names for the cropped images [**.tif** files]
- `center_longitude` (float): center longitude of the fire
- `center_latitude` (float): center latitude of the fire
- `width_km`  (float): full width of box in km to crop centered on coordinate 
- `height_km`  (float): full height of box in km to crop centered on coordinate
<!-- - `radius_in_km` (float): radius from the coordinate in km to crop -->

In [10]:
data_dir = Path('/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths')
tifs = sorted(list(data_dir.rglob('./*HVHV_rtc.grd')))
print(*tifs, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_24082_001_241107_L090_CX_01/fishla_01601_24082_001_241107_HVHV_rtc.grd
/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01/fishla_01601_25026_002_250715_HVHV_rtc.grd
/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25031_010_250922_L090_CX_01/fishla_01601_25031_010_250922_HVHV_rtc.grd
/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_19601_24082_002_241107_L090_CX_01/fishla_19601_24082_002_241107_HVHV_rtc.grd
/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_19601_25026_003_250715_L090_CX_01/fishla_19601_25026_003_250715_HVHV_rtc.grd
/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_19601_25031_009_250922_L090_CX_01/fishla_19601_25031_009_250922_HVHV_rtc.grd


In [13]:
# adjust for wanted and unwanted tifs
include = [
            
          ]
exclude = [
            "241107",
            "001",
            "010",
            "009",
            "003"
          ]

tifs = edit_paths(include,exclude,tifs)
print(*tifs, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01/fishla_01601_25026_002_250715_HVHV_rtc.grd


In [14]:
path_to_images = tifs
print(*path_to_images, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01/fishla_01601_25026_002_250715_HVHV_rtc.grd


In [15]:
# Enter center longitude and latitude for Area of Interest
center_longitude = -111.9894
center_latitude = 38.5835
# radius_in_km = 25  #radius functionality in python/crop_utlis
width_km = 34.99  ## total width and height
height_km = 37.53

output_names = [str(data_dir) + '/' + file.stem + '_box_crop_' + str(width_km) +"km_"+ str(height_km) + 'km.tif' for file in path_to_images]
print(*output_names, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_HVHV_rtc_box_crop_34.99km_37.53km.tif


---
Now perform the cropping and the output will be saved at the working directory. Or you can edit `output_names` variable to set a location

In [16]:
for i in range(len(path_to_images)):
    crop_image_by_coordinates(path_to_images[i], output_names[i],
                              center_longitude, center_latitude,
                              width_km, height_km)

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_HVHV_rtc_box_crop_34.99km_37.53km.tif is outputted!


In [17]:
##data_dir = Path('../data/latuna')
incs = sorted(list(data_dir.rglob('./*.inc')))
# include = [
#           ]
# exclude = [
#           ]

incs = edit_paths(include,exclude,incs)
print(*incs, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01/fishla_01601_25026_002_250715_L090_CX_01.inc


In [18]:
path_to_images = incs
output_names = [str(data_dir) + '/' + file.stem  + '_inc_box_crop_' + str(width_km) +"km_"+ str(height_km) + 'km.tif' for file in path_to_images]
print(*output_names, sep="\n")

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01_inc_box_crop_34.99km_37.53km.tif


---
Now perform the cropping and the output will be saved at the working directory. Or you can edit `output_names` variable to set a location

In [19]:
for i in range(len(path_to_images)):
    crop_image_by_coordinates(path_to_images[i],
                              output_names[i], 
                              center_longitude, 
                              center_latitude, 
                              width_km,
                              height_km)

/mnt/c/Users/PC/Documents/UavsarData/FishLakeFlightPaths/fishla_01601_25026_002_250715_L090_CX_01_inc_box_crop_34.99km_37.53km.tif is outputted!


---
## Crop image by a GeoJSON or Shapefile
Returns a cropped version of the input image corresponding to the polygon(s) in the input GeoJSON or Shapefile.

**Parameters**:
- `path_to_polygon_file` (str): path to the geojson or shapefile
- `path_to_images` (list): a list containing the paths to the images cropping from
- `output_names` (list): a list containing the output names for the cropped images. [**.tif** files]

The sample below will be cropping with a shapefile

In [10]:
# OPTIONAL
# Regressive search directory here 
folder_path = '/Volumes/BlackT7/bobcat_perimeter_wen' # path to shapefiles

shp_dir = Path(folder_path)
shps = sorted(list(shp_dir.glob('./*.shp')))

# OPTIONAL
# adjust for wanted and unwanted shps
# include = [     
#           ]
# exclude = [
#           ]
# shps = edit_paths(include,exclude,shps)

print(*shps, sep="\n")

In [11]:
# Direct enter (NOT OPTIONAL)
# from path or list above
shp = '/Volumes/BlueT7/bobcat_burn_perim/mtbs/2020/ca3424811795920200906/ca3424811795920200906_20200703_20210706_burn_bndy.shp'
shp

'/Volumes/BlueT7/bobcat_burn_perim/mtbs/2020/ca3424811795920200906/ca3424811795920200906_20200703_20210706_burn_bndy.shp'

In [27]:
data_dir = Path('/Volumes/BlackT7/bobcat_flight_paths')
# tifs = sorted(list(data_dir.glob('./*weighted*.tif')))
# tifs = sorted(list(data_dir.rglob('./*_rtc.grd')))

# # adjust for wanted and unwanted tifs
# include = [
#           ]
# exclude = [
#           ]

tifs = edit_paths(include,exclude,tifs)
print(*tifs, sep="\n")

/Volumes/BlackT7/bobcat_flight_paths/SanAnd_08525_18076_003_181011_L090_CX_01/SanAnd_08525_18076_003_181011_HVHV_rtc.grd
/Volumes/BlackT7/bobcat_flight_paths/SanAnd_08525_21041_007_210526_L090_CX_01/SanAnd_08525_21041_007_210526_HVHV_rtc.grd


In [29]:
path_to_polygon_file = shp
path_to_images = tifs
output_names = [str(data_dir) + '/' + file.stem + '_perimeter_crop.tif' for file in tifs]
output_names

['/Volumes/BlackT7/bobcat_flight_paths/SanAnd_08525_18076_003_181011_HVHV_rtc_perimeter_crop.tif',
 '/Volumes/BlackT7/bobcat_flight_paths/SanAnd_08525_21041_007_210526_HVHV_rtc_perimeter_crop.tif']

---
Now perform the cropping and the output will be saved at the working directory. Or you can edit `output_names` variable to set a location

replace `path_to_polygon_file` with the geojson path if cropping by geojson

In [35]:
for i in range(len(path_to_images)):
    crop_image_by_geojson_shp(path_to_polygon_file,
                              path_to_images[i], 
                              output_names[i])

/Volumes/BlackT7/bobcat_flight_paths/SanAnd_08525_20029_006_201015_HVHV_rtc_shp_bndy.tif is outputted.


---
## Reproject Raster image to another coordinate reference systems
Used to reproject the MTBS burn severity map from ESRI:102039 to EPSG:4326, crs of UAVSAR data

The output could be used to extract the burn severity map of a particular fire or area, using `crop_image_by_coordinates`, `crop_image_by_geojson`, or `crop_image_by_shp` from above

**Parameters**
- `target_crs` (CRS): the crs to reproject the image to
- `input_raster_path` (str): path to the original raster image
- `output_raster_path` (str): path for the reprojected raster image [**.tif** files]

In [45]:
data_dir = Path('/Volumes/BlueT7/test_bob/')
tifs = sorted(list(data_dir.glob('./*.tif')))

# adjust for wanted and unwanted tifs
include = [
            "181011"
          ]
exclude = [
            "inc"
          ]

tifs = edit_paths(include,exclude,tifs)
print(*tifs, sep="\n")

/Volumes/BlueT7/test_bob/SanAnd_08525_18076_003_181011_HVHV_rtc_cropped_25km.tif


In [51]:
target_crs = CRS.from_epsg(26911)   
input_raster_path = tifs[0]
output_raster_path = 'SanAnd_08525_18076_003_181011_HVHV_rtc_cropped_25km_crs4326.tif'

---
Now perform the cropping and the output will be saved at the working directory. Or you can edit `output_names` variable to set a location

In [52]:
reproject_geotiff(target_crs, input_raster_path, output_raster_path)

Reprojected raster saved to: SanAnd_08525_18076_003_181011_HVHV_rtc_cropped_25km_crs4326.tif


In [41]:
pwd

'/Users/krismannino/Code/CADS/IMPACT/uavsar-wildfire/notebook'